Quick sanity check: how are different models currently embedding certain users? \
(big matrix only, to prevent leakage)

In [17]:
# Load in big matrix

import deep_linear_bandits.data as dlb_data

DLB_DIR = "/home/sulay/deep-linear-bandits/"

big_train, big_val = dlb_data.preprocess_krbig_interactions(data_dir=DLB_DIR+"kuairec/data/")

In [ ]:
# Get all user & item embeddings for a specific MODEL

import deep_linear_bandits.two_tower as dlb_tt
import numpy as np
import pandas as pd
import torch
import pickle

pd.set_option("display.max_rows", None)
%%javascript
IPython.OutputArea.auto_scroll_threshold = 10;

NUM_USERS = 7176
NUM_ITEMS = 10728

MODEL = "default"

torch.set_float32_matmul_precision('high')
device = (
    torch.accelerator.current_accelerator()
    if torch.accelerator.is_available()
    else torch.device('cpu')
)

with open(DLB_DIR + f"tt-models/{MODEL}/model_args.pkl", "rb") as f:
    model_args = pickle.load(f)
model = dlb_tt.TwoTower(**model_args).to(device)
model.load_state_dict(
    torch.load(
        DLB_DIR + f"tt-models/{MODEL}/model.pt",
        map_location=device,
        weights_only=True
    )
)
model.eval()

with torch.inference_mode():
    """
    Retrieve embeddings for all users & items in the big matrix
    """
    user_ids = torch.arange(0, NUM_USERS, device=device, dtype=torch.long)
    (user_cat_feats, _), user_numeric_feats = dlb_data.preprocess_user_features(data_dir=DLB_DIR+"kuairec/data/")
    user_cat_feats = torch.tensor(user_cat_feats, dtype=torch.long, device=device)
    user_numeric_feats = torch.tensor(user_numeric_feats, dtype=torch.float32, device=device)

    user_embeddings = model.user_tower(user_ids, user_cat_feats, user_numeric_feats).cpu().numpy()

    item_ids = torch.arange(0, NUM_ITEMS, device=device, dtype=torch.long)
    item_categories = dlb_data.preprocess_item_categories(data_dir=DLB_DIR+"kuairec/data/").to(device)

    item_embeddings = model.item_tower(item_ids, item_categories).cpu().numpy()

In [91]:
# What's a specific user U's top items in the big matrix?
# (from training, avoid leakage of val)

U = 1376 # (recall users are between 0 and 7175)

U_wrs = big_train[big_train.user_id == U].sort_values(by='watch_ratio', ascending=False)
U_wrs = U_wrs.rename(columns = {'video_id': 'item_id'})

# What's the model currently got as the top items for that user?
# user_embeddings: (7176, D); for one user it's (D,)
# item_embeddings: (10728, D)
item_scores = item_embeddings @ user_embeddings[U, :]
top_items = np.argsort(item_scores)[::-1]
top_items_scores = item_scores[top_items]

df = pd.DataFrame({
    'item_id': top_items,
    'model_similarity': top_items_scores
})

In [92]:
# Top items by model similarity, and the actual watch_ratio 
df.merge(right=U_wrs, on='item_id').drop(columns=['user_id'])

,item_id,model_similarity,watch_ratio
0,314,0.815631,2.399087
1,4040,0.807161,2.753567
2,8298,0.806486,2.102752
3,5464,0.797069,1.774618
4,2130,0.793370,2.723297
5,4123,0.788099,1.582484
6,2123,0.780243,1.610614
7,154,0.777751,2.291443
8,5434,0.773803,2.527463
9,2894,0.772914,1.399328


In [89]:
# Top items by watch_ratio, and the actual similarity that the model is assigning
U_wrs['model_similarity'] = item_scores[U_wrs['item_id'].values]
U_wrs

,user_id,item_id,watch_ratio,model_similarity
6983321,1376,10184,7.392721,0.497655
4502492,1376,6463,6.188506,0.738533
2163763,1376,3401,4.419501,0.740987
608107,1376,867,3.951360,0.374019
3978056,1376,5738,3.612493,0.417668
...,...,...,...,...
638824,1376,9987,0.000000,0.468134
4187591,1376,7714,0.000000,0.461530
598841,1376,10371,0.000000,0.429262
353559,1376,6177,0.000000,0.505847
